# Trích xuất nhịp thở từ PPG raw (baseline wander method)

Mở rộng của `ppg_hrv_batch_analysis.ipynb` — cùng nguồn PPG raw (`ppg_raw.edf`, 100Hz), cùng cách
xác định phase từ `event_labels_block.csv`, nhưng thay vì lọc băng mạch tâm thu (0.5-8Hz) để tính
HRV, ở đây lọc băng hô hấp (0.1-0.5Hz) để ước lượng nhịp thở/phút và độ đều breath-to-breath.
Chi tiết công thức + citation: xem docstring `lib/ppg_resp.py`.

**Cảnh báo phương pháp**: đây là ước lượng PPG-derived respiration (PDR) bằng baseline-wander,
CHƯA có cảm biến hô hấp tham chiếu để đối chiếu độ chính xác trên bộ dữ liệu này — đọc như tín hiệu
thăm dò, không phải phép đo đã validate lâm sàng.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

REPO_ROOT = Path("..").resolve()
LIB_DIR = REPO_ROOT / "lib"
if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))
import ppg_resp
import importlib
importlib.reload(ppg_resp)

print("ppg_resp module:", ppg_resp.__file__)

In [ ]:
LAYER2_DIR = REPO_ROOT / "data" / "analytics" / "layer2"
LAYER1_SESSIONS_PATH = REPO_ROOT / "data" / "analytics" / "layer1" / "sessions.csv"

OUT_DIR = REPO_ROOT / "data" / "analytics" / "ppg_resp"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EEG_FS = 244.0   # sample rate latency trong event_labels_block.csv (Layer 2) -- giống ppg_hrv_batch_analysis.ipynb
PPG_FS = 100.0   # sample rate ppg_raw.edf

PARTICIPANT_RE = re.compile(r"participant=([^/\\]+)")
STUDY_DAY_RE = re.compile(r"study_day=([^/\\]+)")
SESSION_RE = re.compile(r"session_id=([^/\\]+)")

print("LAYER2_DIR:", LAYER2_DIR, "exists:", LAYER2_DIR.exists())

In [ ]:
def parse_identity(ppg_path: Path) -> dict:
    parts = str(ppg_path)
    participant = PARTICIPANT_RE.search(parts)
    study_day = STUDY_DAY_RE.search(parts)
    session_id = SESSION_RE.search(parts)
    return {
        "participant_key": participant.group(1) if participant else "unknown",
        "study_day": int(study_day.group(1)) if study_day and study_day.group(1).isdigit() else None,
        "session_id": session_id.group(1) if session_id else ppg_path.stem,
    }

ppg_files = sorted(LAYER2_DIR.rglob("ppg_raw.edf"))
print(f"Tìm thấy {len(ppg_files)} file ppg_raw.edf")

sessions_l1 = pd.read_csv(LAYER1_SESSIONS_PATH, low_memory=False) if LAYER1_SESSIONS_PATH.exists() else pd.DataFrame()
role_lookup = (
    sessions_l1.set_index("session_id")["session_role"].to_dict()
    if not sessions_l1.empty and "session_role" in sessions_l1.columns
    else {}
)

session_index = []
for ppg_path in ppg_files:
    identity = parse_identity(ppg_path)
    event_csv = ppg_path.with_name("event_labels_block.csv")
    session_index.append({
        **identity,
        "ppg_path": ppg_path,
        "event_csv_path": event_csv if event_csv.exists() else None,
        "session_role": role_lookup.get(identity["session_id"]),
    })

session_index_df = pd.DataFrame(session_index)
print(f"Có event_labels_block.csv: {session_index_df['event_csv_path'].notna().sum()}/{len(session_index_df)}")

In [ ]:
def analyze_session(ppg_path: Path, event_csv_path, identity: dict) -> list[dict]:
    rows = []
    raw = mne.io.read_raw_edf(ppg_path, preload=True, verbose=False)
    ppg = raw.get_data()[0]
    fs = float(raw.info["sfreq"])

    base_meta = {
        "participant_key": identity["participant_key"],
        "study_day": identity["study_day"],
        "session_id": identity["session_id"],
        "session_role": identity["session_role"],
    }

    overall = ppg_resp.analyze_segment_resp(ppg, fs)
    rows.append({**base_meta, "phase": "OVERALL", **overall})

    if event_csv_path is not None:
        events = pd.read_csv(event_csv_path)
        for _, ev in events.iterrows():
            start_s = ev["latency"] / EEG_FS
            end_s = (ev["latency"] + ev["duration_samples"]) / EEG_FS
            ppg_start = max(0, int(round(start_s * fs)))
            ppg_end = min(len(ppg), int(round(end_s * fs)))
            segment = ppg[ppg_start:ppg_end]

            metrics = ppg_resp.analyze_segment_resp(segment, fs)
            rows.append({
                **base_meta, "phase": ev["description"],
                "phase_start_s": float(start_s), "phase_end_s": float(end_s),
                **metrics,
            })
    return rows

In [ ]:
from tqdm.notebook import tqdm

all_rows = []
run_log = []

for _, row in tqdm(session_index_df.iterrows(), total=len(session_index_df), desc="PPG Resp"):
    identity = row.to_dict()
    try:
        result_rows = analyze_session(row["ppg_path"], row["event_csv_path"], identity)
        all_rows.extend(result_rows)
        run_log.append({"session_id": row["session_id"], "status": "OK", "n_phases": len(result_rows)})
    except Exception as exc:
        run_log.append({"session_id": row["session_id"], "status": "ERROR", "error": f"{type(exc).__name__}: {exc}"})
        print(f"  [ERROR] {row['participant_key']} day={row['study_day']} {row['session_id']}: {exc}")

run_log_df = pd.DataFrame(run_log)
print("\nRun status:")
print(run_log_df["status"].value_counts().to_string())

In [ ]:
ppg_resp_long = pd.DataFrame(all_rows)


def quality_flag(row):
    if pd.isna(row.get("resp_rate_bpm")):
        return "NO_VALID_BREATHS"
    if row.get("segment_duration_s", 0) < 30:
        return "SHORT_SEGMENT"
    if row.get("n_valid_breaths", 0) < 3:
        return "LOW_BREATH_COUNT"
    return "OK"


ppg_resp_long["resp_quality_flag"] = ppg_resp_long.apply(quality_flag, axis=1)

phase_order = ["REST_BEFORE", "MEDITATION", "REST_AFTER", "OVERALL"]
ppg_resp_long["phase"] = pd.Categorical(
    ppg_resp_long["phase"],
    categories=phase_order + sorted(set(ppg_resp_long["phase"]) - set(phase_order)),
    ordered=True,
)
ppg_resp_long = ppg_resp_long.sort_values(["participant_key", "study_day", "phase"]).reset_index(drop=True)

long_path = OUT_DIR / "ppg_resp_by_phase.csv"
ppg_resp_long.to_csv(long_path, index=False)
run_log_df.to_csv(OUT_DIR / "ppg_resp_run_log.csv", index=False)
print(f"Đã lưu: {long_path} ({len(ppg_resp_long)} dòng)")
print()
print("Cờ chất lượng:")
print(ppg_resp_long["resp_quality_flag"].value_counts().to_string())

In [ ]:
metric_cols = ["resp_rate_bpm", "resp_bbi_cv", "n_valid_breaths", "segment_duration_s"]

wide = ppg_resp_long.pivot_table(
    index=["participant_key", "study_day", "session_id"],
    columns="phase", values=metric_cols, observed=True,
)
wide.columns = [f"{metric}__{phase}" for metric, phase in wide.columns]
wide = wide.reset_index()

wide_path = OUT_DIR / "ppg_resp_wide_by_session.csv"
wide.to_csv(wide_path, index=False)
print(f"Đã lưu: {wide_path} ({len(wide)} dòng)")
wide.head(10)

In [ ]:
ok = ppg_resp_long[
    (ppg_resp_long["phase"].isin(["REST_BEFORE", "MEDITATION", "REST_AFTER"]))
    & (ppg_resp_long["resp_quality_flag"] == "OK")
].copy()

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, col, label in zip(axes, ["resp_rate_bpm", "resp_bbi_cv"], ["Nhịp thở (breaths/min)", "Breath-to-breath CV"]):
    data = [ok.loc[ok["phase"] == p, col].dropna() for p in ["REST_BEFORE", "MEDITATION", "REST_AFTER"]]
    ax.boxplot(data, tick_labels=["REST_BEFORE", "MEDITATION", "REST_AFTER"])
    ax.set_title(label)
    ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_DIR / "ppg_resp_by_phase_boxplot.png", dpi=130)
plt.show()

print("N session hợp lệ (resp_quality_flag=OK) theo phase:")
print(ok.groupby("phase", observed=True)["resp_rate_bpm"].apply(lambda s: s.notna().sum()).to_string())
print()
print("Nhịp thở trung vị theo phase (breaths/min):")
print(ok.groupby("phase", observed=True)["resp_rate_bpm"].median().round(1).to_string())